# AI Resume Ranker (ATS Score Checker)

Simple + practical project for AI Intern portfolio.

**Kya karta hai:**
- Resume (PDF ya text) + Job Description daalo
- ATS Match Score nikalta hai
- Missing keywords dikhata hai

Colab pe **Run All** maar do.

In [ ]:
!pip install -q scikit-learn nltk PyPDF2
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print("Libraries installed ✅")

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
import PyPDF2
from google.colab import files

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

print("Functions ready ✅")

### Step 1: Resume Upload karo (PDF) ya baad mein text paste kar dena

In [ ]:
print("Resume PDF upload karo (Cancel dabake text paste bhi kar sakte ho)")
uploaded = files.upload()

resume_text = ""
if uploaded:
    for filename in uploaded.keys():
        resume_text = extract_text_from_pdf(filename)
        print(f"\n✅ Extracted from: {filename}")
        print("Preview:", resume_text[:300], "...")
else:
    print("PDF nahi daali. Neeche text paste karna.")

### Step 2: Agar PDF nahi daali to yahan resume text paste karo

In [ ]:
if not resume_text.strip():
    resume_text = input("Resume text yahan paste karo:\n")
print("\nResume length:", len(resume_text), "characters")

### Step 3: Job Description paste karo

In [ ]:
job_description = input("Job Description yahan paste karo:\n")
print("\nJD length:", len(job_description), "characters")

### Step 4: Score nikalta hai

In [ ]:
clean_resume = clean_text(resume_text)
clean_jd = clean_text(job_description)

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([clean_jd, clean_resume])
score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

jd_words = set(clean_jd.split())
resume_words = set(clean_resume.split())
missing = list(jd_words - resume_words)

print("="*50)
print(f"🔥 ATS MATCH SCORE : {round(score * 100, 2)}%")
print("="*50)
print("\nMissing important keywords (top 15):")
print(", ".join(missing[:15]) if missing else "Koi major keyword missing nahi!")
print("\n💡 Tip: In keywords ko naturally resume mein add karo.")